# 4. Estimator properties



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, expon
from scipy.optimize import minimize

rng = np.random.default_rng(seed=123)

# Problem setup
# ------- true parameter values -------
mu_true     = 7.0   # Gaussian mean
sigma_true  = 1.0   # Gaussian width

mu_bias = 9.0
sigma_bias = 1.0

mu_variance = 7.0
sigma_variance = 3.0

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

xx = np.linspace(0, 17, 500)
pdf_true = (norm.pdf(xx, mu_true, sigma_true))
pdf_bias = (norm.pdf(xx, mu_bias, sigma_bias))
pdf_variance = (norm.pdf(xx, mu_variance, sigma_variance))

ax.plot(xx, pdf_true, 'r-', lw=2, label='Best estimator')
ax.plot(xx, pdf_bias, 'g-', lw=2, label='Biased estimator')
ax.plot(xx, pdf_variance, 'b-', lw=2, label='Large-variance estimator')

# True parameter position
ax.axvline(mu_true, color='k', linestyle='--', lw=1.8)
ax.annotate(r'$\theta$', xy=(mu_true, 0), xytext=(0, -26),
            textcoords='offset points', ha='center', va='top', fontsize=16,
            annotation_clip=False)

ax.set_xlabel('$\\hat{\\theta}$', loc='right', fontsize=16)
ax.set_ylabel('Probability density', loc='top', fontsize=16)
ax.set_title('')
ax.legend(frameon=False, fontsize=16)
plt.tight_layout()
plt.show()

# 9. Maximum likelihood estimators


In [ ]:
# Problem setup
mu_true     = 7.0   # Gaussian mean
sigma_true  = 1.0   # Gaussian width

mu_bias = 9.0
sigma_bias = 1.0

mu_variance = 7.0
sigma_variance = 3.0

In [ ]:
# ------- true parameter values -------
mu_true     = 7.0   # Gaussian mean
sigma_true  = 1.0   # Gaussian width

n_samples = 2000    # total number of events

# ------- sampling -------
x_data  = rng.normal(mu_true, sigma_true, size=n_samples)

print(f"Generated {len(xx)} events")

# MLE for Gaussian mean is the sample mean
mu_mle = x_data.mean()
sigma_mle = x_data.std(ddof=0)  # MLE estimate of sigma

mu_bias = x_data.mean() - 1
sigma_bias = x_data.std(ddof=0)  # MLE estimate of sigma

mu_variance = x_data.mean()
sigma_variance = x_data.std(ddof=0) * 3.0

In [ ]:
xx = np.linspace(0, 14, 600)

pdf_true = (norm.pdf(xx, mu_true, sigma_true))
pdf_mle = (norm.pdf(xx, mu_mle, sigma_mle))
pdf_bias = (norm.pdf(xx, mu_bias, sigma_bias))
pdf_variance = (norm.pdf(xx, mu_variance, sigma_variance)) 

fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(x_data, bins=60, range=(0, 14), density=True,
        histtype='stepfilled', alpha=0.3, color='steelblue', label='Data')
ax.plot(xx, pdf_true, 'k--', lw=2,  label='True PDF')
ax.plot(xx, pdf_mle, 'r-',  lw=2,  label='MLE Estimator')
ax.plot(xx, pdf_bias, 'g-', lw=2, label='Non-MLE Estimator 1')
ax.plot(xx, pdf_variance, 'b-', lw=2, label='Non-MLE Estimator 2')
ax.set_xlim(2, 14)

ax.set_xlabel('$x$', loc='right', fontsize=16)
ax.set_ylabel('$f(x)$', loc='top', fontsize=16)
ax.set_title('')
ax.legend(frameon=False, fontsize=16)
plt.tight_layout()
plt.show()

# 9. Distribution of MLE becomes Gaussian

In [ ]:
# ------- true parameter values -------
lambda_true = 0.5   # exponential rate  (mean = 1/lambda)

n_samples = 500    # total number of events
n_repeats = 50   # number of repeated experiments

# ------- sampling -------
theta_hats = []
for _ in range(n_repeats):
    x_data = rng.exponential(1.0 / lambda_true, size=n_samples)
    tau_mle = x_data.mean()
    theta_hats.append(tau_mle)

In [ ]:
xx = np.linspace(1.5, 2.5, 600)

# Fit a Gaussian to the repeated-estimator samples
mu_fit, sigma_fit = norm.fit(theta_hats)
pdf_fit = norm.pdf(xx, loc=mu_fit, scale=sigma_fit)

fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(theta_hats, bins=25, range=(1.5, 2.5), density=True,
        histtype='stepfilled', alpha=0.3, color='steelblue', label='MLE estimate')
ax.plot(xx, pdf_fit, 'r-', lw=2, label='Gaussian fit')
ax.set_xlim(1.5, 2.5)
ax.text(0.03, 0.95, f'Pseudo-experiments: {n_repeats}',
        transform=ax.transAxes, ha='left', va='top', fontsize=13)

ax.set_xlabel('$\\hat{\\tau}$', loc='right', fontsize=16)
ax.set_ylabel('$f(\\hat{\\tau})$', loc='top', fontsize=16)
ax.set_title('')
ax.legend(frameon=False, fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
print(f"Mean of MLE estimates: {mu_fit:.3f}")
print(f"Standard deviation of MLE estimates: {sigma_fit:.3f}")

# Least squares stuff

Generate 8 independent measurements, equally distributed between 1 and 9 of a quantity y = 0.5 * x and apply a gaussian smearing to the y value with a sigma of 1. 
Assign to each measurement a y uncertainty distributed as a gaussian with mean 0.5 and sigma 0.2.
Make a plot of y vs x, fit a 3rd order polinomial to the data and plot the fitted function with a blue line 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Reproducible random generator for this section
rng_ls = np.random.default_rng(seed=123)

# 8 independent measurements with x equally spaced in [1, 9]
x = np.linspace(1, 9, 16)
y_true = 0.5 * x

# Gaussian smearing on y with sigma = 1
y = y_true + rng_ls.normal(0.0, 1.0, size=x.size)

# Per-point y uncertainty from N(0.5, 0.3), clipped to keep physical positive errors
yerr = rng_ls.normal(0.5, 0.1, size=x.size)
yerr = np.clip(yerr, 0.05, None)

# Weighted cubic fit (weights proportional to 1/sigma_y)
coeffs = np.polyfit(x, y, deg=3, w=1.0 / yerr)
poly3 = np.poly1d(coeffs)

xx = np.linspace(1, 9, 400)
yy = poly3(xx)

fig, ax = plt.subplots(figsize=(8, 6))
ax.errorbar(x, y, yerr=yerr, fmt='o', ms=6, color='black', ecolor='gray',
            capsize=3, label='Measurements')
ax.plot(xx, yy, 'b-', lw=2.5, label='f(x, $\\theta$)')

ax.set_xlabel('$x$', loc='right', fontsize=16)
ax.set_ylabel('$y$', loc='top', fontsize=16)
ax.set_title('')
ax.legend(frameon=False, fontsize=16)
ax.set_xlim(0, 10.)
plt.tight_layout()
plt.show()

print('Cubic fit coefficients (highest degree first):')
print(coeffs)

In [ ]:
# Good fit

# Reproducible random generator for this section
rng_ls = np.random.default_rng(seed=24)

# 8 independent measurements with x equally spaced in [1, 9]
x = np.linspace(1, 9, 8)
y_true = 0.5 * x

# Gaussian smearing on y with sigma = 0.7
y = y_true + rng_ls.normal(0.0, 0.7, size=x.size)

# Per-point y uncertainty from N(0.5, 0.3), clipped to keep physical positive errors
yerr = rng_ls.normal(0.5, 0.1, size=x.size)
yerr = np.clip(yerr, 0.05, None)

# Weighted linear fit (weights proportional to 1/sigma_y)
coeffs = np.polyfit(x, y, deg=1, w=1.0 / yerr)
poly1 = np.poly1d(coeffs)

# Goodness-of-fit for linear model
residuals = (y - poly1(x)) / yerr
chi2 = np.sum(residuals**2)
ndf = x.size - 2  # n_points - n_parameters

xx = np.linspace(1, 9, 400)
yy = poly1(xx)

fig, ax = plt.subplots(figsize=(8, 6))
ax.errorbar(x, y, yerr=yerr, fmt='o', ms=6, color='black', ecolor='gray',
            capsize=3, label='Measurements')
ax.plot(xx, yy, 'b-', lw=2.5, label='f(x, $\\theta$)')
ax.text(0.03, 0.97, f'$\\chi^2$ = {chi2:.2f} \nndf = {ndf} \n$\\chi^2$/ndf = {chi2 / ndf:.2f}',
        transform=ax.transAxes, ha='left', va='top', fontsize=12,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='none'))

ax.set_xlabel('$x$', loc='right', fontsize=16)
ax.set_ylabel('$y$', loc='top', fontsize=16)
ax.set_title('')
ax.legend(frameon=False, fontsize=16)
ax.set_xlim(0, 10.)
plt.tight_layout()
plt.show()

# Chi-square distribution for this ndf, with observed chi2 highlighted
from scipy.stats import chi2 as chi2_dist
chi2_xmax = max(chi2 * 1.3, ndf + 6 * np.sqrt(2 * ndf))
chi2_x = np.linspace(0.0, chi2_xmax, 600)
chi2_pdf = chi2_dist.pdf(chi2_x, df=ndf)
p_value = chi2_dist.sf(chi2, df=ndf)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(chi2_x, chi2_pdf, color='darkorange', lw=2.5, label=fr'$\chi^2$ PDF (ndf={ndf})')
tail_mask = chi2_x >= chi2
ax.fill_between(chi2_x[tail_mask], chi2_pdf[tail_mask], 0.0,
                color='tomato', alpha=0.35, label=fr'Right tail (p = {p_value:.3f})')
ax.axvline(chi2, color='crimson', linestyle='--', lw=2, label=fr'Observed $\chi^2$ = {chi2:.2f}')
ax.text(0.97, 0.95, f'p-value = {p_value:.3f}',
        transform=ax.transAxes, ha='right', va='top', fontsize=12,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='none'))
ax.set_xlabel('$\chi^2_{min}$', loc='right', fontsize=16)
ax.set_ylabel('Probability density', loc='top', fontsize=16)
ax.set_title('')
ax.legend(frameon=False, fontsize=13)
plt.tight_layout()
plt.show()

print('Linear fit coefficients (highest degree first):')
print(coeffs)
print(f'chi2 = {chi2:.3f}')
print(f'ndf = {ndf}')
print(f'chi2/ndf = {chi2 / ndf:.3f}')
print(f'p-value = {p_value:.4f}')

In [ ]:
# Bad fit

# Reproducible random generator for this section
rng_ls = np.random.default_rng(seed=17)

# 8 independent measurements with x equally spaced in [1, 9]
x = np.linspace(1, 9, 8)
y_true = 0.5 * x

# Gaussian smearing on y with sigma = 1
y = y_true + rng_ls.normal(0.0, 0.8, size=x.size)

# Per-point y uncertainty from N(0.5, 0.3), clipped to keep physical positive errors
yerr = rng_ls.normal(0.5, 0.1, size=x.size)
yerr = np.clip(yerr, 0.05, None)

# Weighted linear fit (weights proportional to 1/sigma_y)
coeffs = np.polyfit(x, y, deg=1, w=1.0 / yerr)
poly1 = np.poly1d(coeffs)

# Goodness-of-fit for linear model
residuals = (y - poly1(x)) / yerr
chi2 = np.sum(residuals**2)
ndf = x.size - 2  # n_points - n_parameters

xx = np.linspace(1, 9, 400)
yy = poly1(xx)

fig, ax = plt.subplots(figsize=(8, 6))
ax.errorbar(x, y, yerr=yerr, fmt='o', ms=6, color='black', ecolor='gray',
            capsize=3, label='Measurements')
ax.plot(xx, yy, 'b-', lw=2.5, label='f(x, $\\theta$)')
ax.text(0.03, 0.97, f'$\\chi^2$ = {chi2:.2f} \nndf = {ndf} \n$\\chi^2$/ndf = {chi2 / ndf:.2f}',
        transform=ax.transAxes, ha='left', va='top', fontsize=12,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='none'))

ax.set_xlabel('$x$', loc='right', fontsize=16)
ax.set_ylabel('$y$', loc='top', fontsize=16)
ax.set_title('')
ax.legend(frameon=False, fontsize=16)
ax.set_xlim(0, 10.)
plt.tight_layout()
plt.show()

# Chi-square distribution for this ndf, with observed chi2 highlighted
from scipy.stats import chi2 as chi2_dist
chi2_xmax = max(chi2 * 1.3, ndf + 6 * np.sqrt(2 * ndf))
chi2_x = np.linspace(0.0, chi2_xmax, 600)
chi2_pdf = chi2_dist.pdf(chi2_x, df=ndf)
p_value = chi2_dist.sf(chi2, df=ndf)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(chi2_x, chi2_pdf, color='darkorange', lw=2.5, label=fr'$\chi^2$ PDF (ndf={ndf})')
tail_mask = chi2_x >= chi2
ax.fill_between(chi2_x[tail_mask], chi2_pdf[tail_mask], 0.0,
                color='tomato', alpha=0.35, label=fr'Right tail (p = {p_value:.3f})')
ax.axvline(chi2, color='crimson', linestyle='--', lw=2, label=fr'Observed $\chi^2$ = {chi2:.2f}')
ax.text(0.97, 0.95, f'p-value = {p_value:.3f}',
        transform=ax.transAxes, ha='right', va='top', fontsize=12,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8, edgecolor='none'))
ax.set_xlabel('$\chi^2$', loc='right', fontsize=16)
ax.set_ylabel('Probability density', loc='top', fontsize=16)
ax.set_title('')
ax.legend(frameon=False, fontsize=13)
plt.tight_layout()
plt.show()

print('Linear fit coefficients (highest degree first):')
print(coeffs)
print(f'chi2 = {chi2:.3f}')
print(f'ndf = {ndf}')
print(f'chi2/ndf = {chi2 / ndf:.3f}')
print(f'p-value = {p_value:.4f}')

# ML and bayesian estimators

In [ ]:
rng = np.random.default_rng(seed=123)

# Problem setup
# ------- true parameter values -------
mu_nom     = 7.0   # Gaussian mean
sigma_nom  = 1.0   # Gaussian width

mu_2 = 2.0
sigma_2 = 5.0

mu_3 = 11.0
sigma_3 = 3.0

fig, ax = plt.subplots(figsize=(8, 6))

xx = np.linspace(0, 17, 500)
pdf_nom = (norm.pdf(xx, mu_nom, sigma_nom) +
           norm.pdf(xx, mu_2, sigma_2) +
           2.*norm.pdf(xx, mu_3, sigma_3))

# Mode of the estimator distribution (numerical maximum on plotting grid)
mode_x = xx[np.argmax(pdf_nom)]

ax.plot(xx, pdf_nom, 'r-', lw=2, label='Best estimator')

# Mode position
ax.axvline(mode_x, color='k', linestyle='--', lw=1.8)
ax.annotate(r'$\hat{\theta}_{\mathrm{Bayes}}$', xy=(mode_x, 0), xytext=(0, -26),
            textcoords='offset points', ha='center', va='top', fontsize=16,
            annotation_clip=False)

ax.set_xlabel('$\\theta$', loc='right', fontsize=16)
ax.set_ylabel('Probability density', loc='top', fontsize=16)
ax.set_title('')
#ax.legend(frameon=False, fontsize=16)
plt.tight_layout()
plt.show()